# 🎬 Cinematch - Fase 1: Configuración de Carpetas, Entrenamiento y Evaluación del Modelo (IDL2)

En este cuaderno realizaremos la preparación del proyecto y el modelado predictivo con la data limpia de **Cinematch** (682 registros de películas):

1. Conexión a Google Drive y creación automática de la estructura de carpetas de software.
2. Sincronización inicial con nuestro repositorio en GitHub.
3. Carga y preparación del dataset depurado en el IDL1 (`data_limpia_peliculas_knime.tab`).
4. División de datos en Entrenamiento (80%) y Prueba (20%).
5. Entrenamiento de modelos Random Forest y cálculo de métricas cuantitativas (Exactitud, Precisión, Exhaustividad, F1-Score y MAE).
6. Análisis de Overfitting vs Underfitting mediante la variación del número de árboles (20, 50, 100 y 200).
7. Guardado del modelo entrenado (`.pkl`) y subida automática de resultados a GitHub.


**Objetivos principales de este cuaderno:**
1. Crear automáticamente la arquitectura de carpetas en Google Drive.
2. Cargar y depurar el dataset limpio de IDL1 (`data_limpia_peliculas_knime.tab`).
3. Dividir los datos en conjuntos de Entrenamiento (80%) y Prueba (20%).
4. Entrenar un modelo de **Random Forest Classifier** regularizado para evitar el sobreajuste (*Overfitting*).
5. Evaluar el desempeño mediante métricas cuantitativas (Exactitud, Precisión, Exhaustividad, F1-Score y MAE).
6. Analizar el comportamiento del error al variar la profundidad del árbol.
7. Exportar el modelo entrenado (`.pkl`) y sincronizar los avances en GitHub.

In [ ]:
# ==============================================================================
# I. Conexión a Google Drive y Creación de la Estructura de Carpetas
# ==============================================================================

import os
from google.colab import drive

drive.mount('/content/drive')

ROOT_DIR = '/content/drive/MyDrive/Cinematch_IDL2_DS_IA'

subcarpetas = [
    'backend/models',
    'backend/api',
    'frontend/assets',
    'db',
    'docs/diagrams',
    'docs/metrics',
    'docs/report_drafts',
]

print('Creando estructura limpia de carpetas en Google Drive...')
for carpeta in subcarpetas:
  ruta_completa = os.path.join(ROOT_DIR, carpeta)
  os.makedirs(ruta_completa, exist_ok=True)
  print(f'   Carpeta lista: {carpeta}')

readme_contenido = """# Cinematch IDL2 - Sistema Inteligente de Clasificación de Películas

Proyecto desarrollado para la unidad didáctica Desarrollo de Soluciones con IA.

## Estructura de Carpetas
- `DATASET/`: Dataset depurado en IDL1 (682 registros).
- `notebooks/`: Cuadernos de experimentación y entrenamiento en Google Colab.
- `backend/`: Modelo entrenado (.pkl) y servicios API REST.
- `frontend/`: Aplicación web interactiva construida en Streamlit.
- `db/`: Base de datos SQLite y scripts de diseño (DER).
- `docs/`: Gráficos de evaluación de métricas y borradores del informe final.
"""

with open(os.path.join(ROOT_DIR, 'README.md'), 'w', encoding='utf-8') as f:
  f.write(readme_contenido)

with open(os.path.join(ROOT_DIR, '.gitignore'), 'w', encoding='utf-8') as f:
  f.write('__pycache__/\n*.pyc\n.DS_Store\n*.tmp\n')

print('\n Estructura de software generada con éxito!')

### **Vincular el Proyecto con GitHub**
Configuramos las credenciales de Git y vinculamos nuestro espacio de trabajo con el repositorio remoto de GitHub (`Cinematch_IDL2_DS_IA-`). Esto garantiza el control de versiones y respalda los archivos de código, modelos y gráficos en la nube.

In [ ]:
# ==============================================================================
# II. Configuración e Inicialización de Git
# ==============================================================================

import os
from google.colab import userdata

try:
  GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
  from getpass import getpass

  GITHUB_TOKEN = getpass('Ingresar GitHub Personal Access Token: ')

GITHUB_USER = 'Yaseli-M'
GITHUB_EMAIL = 'i2511821@continental.edu.pe'
GITHUB_REPO = 'Cinematch_IDL2_DS_IA-'

%cd /content/drive/MyDrive/Cinematch_IDL2_DS_IA

!git config --global user.name "{GITHUB_USER}"
!git config --global user.email "{GITHUB_EMAIL}"

!git init
!git branch -M main
!git add .
!git commit -m "Commit 1: Estructura inicial del proyecto Cinematch"

!git remote remove origin 2>/dev/null || true
!git remote add origin https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git
!git push -u origin main --force

### **Carga del Dataset Limpio**

Cargamos el archivo `data_limpia_peliculas_knime.tab` almacenado en la carpeta `DATASET/`. Este archivo contiene las 682 películas procesadas durante el IDL1 con variables como género, duración, nivel de violencia, romance, animación, rating y fecha de estreno.



In [ ]:
# ==============================================================================
# III. Carga y Exploración de Datos
# ==============================================================================

import pandas as pd
import numpy as np

# 1. Definir la ruta del dataset
data_path = os.path.join(ROOT_DIR, 'DATASET', 'data_limpia_peliculas_knime.tab')

# 2. Leer el archivo
try:
    df = pd.read_csv(data_path, sep='\t')
except Exception:
    df = pd.read_csv(data_path)

print(f"Dataset cargado correctamente. Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas.\n")

# 3. Mostrar primeras 5 filas
df.head()

### **División de Datos: Entrenamiento (80%) y Prueba (20%)**

Aplicamos el **Train/Test Split**.
- **80% para Entrenamiento (Train)**
- **20% para Prueba (Test)**

In [ ]:
# ==============================================================================
# IV. Preprocesamiento y División Train / Test
# ==============================================================================

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df_prep = df.copy()

# 1. Normalizar equivalentes a texto nulo
df_prep = df_prep.replace(['NAN', '?', 'nan', 'NaN'], np.nan)

# 2. Descartar únicamente los registros donde falta la etiqueta objetivo
df_prep = df_prep.dropna(subset=['publico_objetivo']).copy()

# 3. Procesar fecha de estreno (extraer año) e imputar con la mediana
if 'fecha_estreno' in df_prep.columns:
  df_prep['anio_estreno'] = pd.to_datetime(
      df_prep['fecha_estreno'], errors='coerce'
  ).dt.year
  df_prep['anio_estreno'] = df_prep['anio_estreno'].fillna(
      df_prep['anio_estreno'].median()
  )

# 4. Convertir duración a numérico e imputar nulos con la mediana
if 'duracion' in df_prep.columns:
  df_prep['duracion'] = pd.to_numeric(df_prep['duracion'], errors='coerce')
  df_prep['duracion'] = df_prep['duracion'].fillna(df_prep['duracion'].median())

# 5. Separar características predictoras (X) y target (y)
cols_inutiles = [
    'Selected',
    'id',
    'titulo',
    'fecha_estreno',
    'publico_objetivo',
]
cols_a_borrar = [c for c in cols_inutiles if c in df_prep.columns]

X = df_prep.drop(columns=cols_a_borrar)
y = df_prep['publico_objetivo']

# 6. Imputar variables categóricas con la moda
for col in X.select_dtypes(include=['object']).columns:
  if X[col].isnull().sum() > 0:
    X[col] = X[col].fillna(X[col].mode()[0])

# 7. Codificación binaria (One-Hot Encoding) y codificación de la variable objetivo
X_encoded = pd.get_dummies(X, drop_first=True)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 8. División 80% Entrenamiento / 20% Prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

print('     PREPROCESAMIENTO Y DIVISIÓN DE DATOS         ')
print('==================================================')
print(f' Muestra limpia para modelar: {X.shape[0]} películas (643 de 682).')
print(
    f' Clases reales del target ({len(label_encoder.classes_)}):'
    f' {list(label_encoder.classes_)}'
)
print(f' Datos de Entrenamiento (Train): {X_train.shape[0]} películas')
print(f' Datos de Prueba (Test):        {X_test.shape[0]} películas')

### **Entrenamiento del Modelo y Métricas de Rendimiento**

Métricas de clasificación:
- **Exactitud (Accuracy)**: Porcentaje general de aciertos en la categoría del público.
- **Precisión (Precision)** y **Exhaustividad (Recall)**: Calidad de la detección por clase.
- **F1-Score**: Promedio armónico entre Precisión y Exhaustividad.
- **MAE**: Error absoluto medio entre las etiquetas codificadas.

In [ ]:
# ==============================================================================
# V. Entrenamiento del Modelo y Cálculo de Métricas de Clasificación
# ==============================================================================

import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# 1. Entrenar Random Forest
modelo_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_split=10,
    random_state=42,
    class_weight='balanced',
)
modelo_rf.fit(X_train, y_train)

# 2. Generar predicciones
y_pred = modelo_rf.predict(X_test)

# 3. Métricas de Clasificación
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

# 4. Tabla de Métricas
tabla_metricas = pd.DataFrame({
    'Métrica de Clasificación': [
        'Exactitud (Accuracy)',
        'Precisión (Precision)',
        'Exhaustividad (Recall)',
        'F1-Score',
    ],
    'Valor Obtenido': [
        round(acc, 4),
        round(prec, 4),
        round(rec, 4),
        round(f1, 4),
    ],
})

print('   RESULTADOS DE EVALUACIÓN DEL MODELO CINEMATCH  ')
print('==================================================')
print(tabla_metricas.to_string(index=False))

# Guardar la tabla
tabla_metricas.to_csv(
    os.path.join(ROOT_DIR, 'docs/report_drafts', 'tabla_metricas_modelos.csv'),
    index=False,
)

# 5. Generar y guardar Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
)
plt.title('Matriz de Confusión - Público Objetivo (Cinematch)')
plt.xlabel('Clase Predicha')
plt.ylabel('Clase Real')
plt.tight_layout()
plt.savefig(
    os.path.join(ROOT_DIR, 'docs/metrics', 'matriz_confusion.png'), dpi=300
)
plt.show()

### **Experimento: Análisis de Overfitting vs Underfitting (Variación de Árboles)**

Variamos el número de estimadores/árboles en Random Forest (**20, 50, 100 y 200 árboles**) para observar cómo cambia el error entre Entrenamiento y Prueba, identificando el punto de ajuste óptimo.

In [ ]:
# ==============================================================================
# VI. Experimento Overfitting vs Underfitting y Gráfica
# ==============================================================================

import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Rango de profundidades
profundidades = list(range(1, 11))
errores_train = []
errores_test = []

for depth in profundidades:
  model = RandomForestClassifier(
      n_estimators=100,
      max_depth=depth,
      random_state=42,
      class_weight='balanced',
  )
  model.fit(X_train, y_train)

  # Tasa de error = 1.0 - Accuracy
  err_tr = 1.0 - accuracy_score(y_train, model.predict(X_train))
  err_te = 1.0 - accuracy_score(y_test, model.predict(X_test))

  errores_train.append(err_tr)
  errores_test.append(err_te)

# 2. Graficar curvas de error (Train vs Test)
plt.figure(figsize=(8, 5))
plt.plot(
    profundidades,
    errores_train,
    label='Tasa Error Entrenamiento (Train)',
    marker='o',
    color='blue',
)
plt.plot(
    profundidades,
    errores_test,
    label='Tasa Error Prueba (Test)',
    marker='s',
    color='red',
)
plt.title('Análisis de Overfitting vs Underfitting (Variación de max_depth)')
plt.xlabel('Profundidad Máxima del Árbol (max_depth)')
plt.ylabel('Tasa de Error (1 - Exactitud)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

grafico_path = os.path.join(
    ROOT_DIR, 'docs/metrics', 'overfitting_vs_underfitting.png'
)
plt.savefig(grafico_path, dpi=300, bbox_inches='tight')
plt.show()

print(f' Gráfico guardado exitosamente en: {grafico_path}')

### **Guardado del Modelo Entrenado y Sincronización Final con GitHub**

Guardamos el modelo entrenado (`.pkl`) dentro de la carpeta `backend/models/`. Finalmente, guardamos todo (tablas, gráficos y modelo) en GitHub.

In [ ]:
# ==============================================================================
# VII. Guardar Modelo (.pkl) y Actualizar GitHub
# ==============================================================================

import os
import joblib

# 1. Exportar el modelo entrenado y el encoder
modelo_export_path = os.path.join(
    ROOT_DIR, 'backend/models', 'modelo_cinematch.pkl'
)
encoder_export_path = os.path.join(
    ROOT_DIR, 'backend/models', 'label_encoder.pkl'
)

joblib.dump(modelo_rf, modelo_export_path)
joblib.dump(label_encoder, encoder_export_path)

print(f' Modelo exportado exitosamente en: {modelo_export_path}')
print(f' Encoder exportado exitosamente en: {encoder_export_path}')

# 2. Sincronizar todos los cambios con el repositorio de GitHub
%cd /content/drive/MyDrive/Cinematch_IDL2_DS_IA

!git add .
!git commit -m "Commit 2: Modelo regularizado, métricas de clasificación y gráficos actualizados"

!git remote remove origin 2>/dev/null || true
!git remote add origin https://{GITHUB_TOKEN}@github.com/Yaseli-M/Cinematch_IDL2_DS_IA-.git
!git push origin main --force

print(
    '\n Proyecto subido con éxito a repo de GitHub!'
)